In [1]:
import pandas as pd
from pathlib import Path

# =============================================================================
# PATHS
# =============================================================================

DATA_DIR = Path("..") / "data"
DEPLOY_DIR = DATA_DIR / "deployment"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = DATA_DIR / "district_final_risk_score.csv"
OUTPUT_CSV = DEPLOY_DIR / "district_final_risk_score_transposed.csv"

df = pd.read_csv(INPUT_CSV)
print("Loaded:", INPUT_CSV, "shape:", df.shape)

ID_COL = "object-id" if "object-id" in df.columns else "object_id"
TIME_COL = "timeperiod"

# =============================================================================
# MELT TO LONG FORMAT (one row per id x timeperiod x factor)
# =============================================================================

numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
id_vars = [ID_COL, TIME_COL]
df_numeric = df[id_vars + numeric_columns]

excluded_columns = [c for c in df.columns if c not in df_numeric.columns]
print("Excluded (non-numeric, non-id) columns:", excluded_columns)

df_melted = df_numeric.melt(id_vars=id_vars, var_name="factor", value_name="score")
print("Melted shape:", df_melted.shape)

# =============================================================================
# PIVOT TO WIDE FORMAT (one column per id, one row per factor x timeperiod)
# =============================================================================

df_transposed = df_melted.pivot(
    index=["factor", TIME_COL], columns=ID_COL, values="score"
).reset_index()
print("Transposed shape:", df_transposed.shape)

df_transposed.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)

# =============================================================================
# SANITY CHECK
# (pivoted value for a sample id/timeperiod/factor must match the source)
# =============================================================================

sample = df_melted.dropna(subset=["score"]).iloc[0]

transposed_value = df_transposed.loc[
    (df_transposed["factor"] == sample["factor"])
    & (df_transposed[TIME_COL] == sample[TIME_COL]),
    sample[ID_COL],
].iloc[0]

assert transposed_value == sample["score"], (
    f"Transpose mismatch for factor={sample['factor']!r}, "
    f"{TIME_COL}={sample[TIME_COL]!r}, {ID_COL}={sample[ID_COL]!r}: "
    f"expected {sample['score']}, got {transposed_value}"
)
print("Sanity check passed: transposed value matches source for a sample row")

Loaded: ../data/district_final_risk_score.csv shape: (1890, 29)
Excluded (non-numeric, non-id) columns: ['district', 'dtname', 'land-surface-temperature-raster']
Melted shape: (45360, 4)
Transposed shape: (1512, 32)
Saved: ../data/deployment/district_final_risk_score_transposed.csv
Sanity check passed: transposed value matches source for a sample row
